# Python `functools` Module

`functools` provides higher-order functions — tools that work **on** other functions.  
These are heavily used in data pipelines, caching, and functional-style Python.

| Function | Purpose |
|----------|---------|
| `reduce()` | Fold a sequence into a single value |
| `partial()` | Fix some arguments of a function, create a new one |
| `lru_cache()` | Cache function results to avoid recomputation |
| `wraps()` | Preserve metadata when writing decorators |

## 1. `reduce()` — Fold a Sequence into One Value

`reduce(func, iterable)` applies `func` cumulatively to pairs of items from left to right until one value remains.

```
reduce(f, [a, b, c, d])  →  f(f(f(a, b), c), d)
```

In [1]:
from functools import reduce

# Sum — reduce vs built-in sum()
numbers = [1, 2, 3, 4, 5]

total = reduce(lambda acc, x: acc + x, numbers)
print("Sum via reduce :", total)          # 15
print("Sum via built-in:", sum(numbers))  # 15 — prefer this for simple sums

# Product — no built-in, so reduce shines here
product = reduce(lambda acc, x: acc * x, numbers)
print("Product        :", product)        # 120

# Finding max without max()
biggest = reduce(lambda a, b: a if a > b else b, numbers)
print("Max via reduce :", biggest)        # 5

# Flatten a list of lists
nested = [[1, 2], [3, 4], [5, 6]]
flat   = reduce(lambda acc, x: acc + x, nested)
print("Flattened      :", flat)           # [1, 2, 3, 4, 5, 6]

# With initial value (3rd argument) — safe on empty iterables
empty   = []
default = reduce(lambda acc, x: acc + x, empty, 0)
print("Empty with init:", default)        # 0

Sum via reduce : 15
Sum via built-in: 15
Product        : 120
Max via reduce : 5
Flattened      : [1, 2, 3, 4, 5, 6]
Empty with init: 0


In [2]:
from functools import reduce

# Real-world: compute total salary from employee records
employees = [
    {"name": "Alice",   "salary": 95000},
    {"name": "Bob",     "salary": 80000},
    {"name": "Charlie", "salary": 70000},
]

total_salary = reduce(lambda acc, emp: acc + emp["salary"], employees, 0)
print("Total payroll: $", total_salary)   # $245,000

# Build a sentence from words (same as " ".join but shows reduce)
words    = ["Python", "is", "great"]
sentence = reduce(lambda a, b: a + " " + b, words)
print("Sentence:", sentence)

Total payroll: $ 245000
Sentence: Python is great


## 2. `partial()` — Pre-fill Function Arguments

`partial(func, *args, **kwargs)` returns a new function with some arguments already fixed.  
Great for creating specialised versions of general functions.

In [3]:
from functools import partial

# Base function
def power(base, exponent):
    return base ** exponent

# Create specialised versions
square = partial(power, exponent=2)
cube   = partial(power, exponent=3)

print("square(5):", square(5))   # 25
print("cube(3)  :", cube(3))     # 27

# partial with positional args
def greet(greeting, name):
    return f"{greeting}, {name}!"

say_hello = partial(greet, "Hello")
say_hi    = partial(greet, "Hi")

print(say_hello("Alice"))   # Hello, Alice!
print(say_hi("Bob"))        # Hi, Bob!

# Apply to a list using map
names  = ["Alice", "Bob", "Charlie"]
greets = list(map(say_hello, names))
print(greets)

square(5): 25
cube(3)  : 27
Hello, Alice!
Hi, Bob!
['Hello, Alice!', 'Hello, Bob!', 'Hello, Charlie!']


In [4]:
from functools import partial

# Real-world: pre-configured logging function
def log(level, service, message):
    print(f"[{level}] [{service}] {message}")

# Create service-specific loggers
api_log  = partial(log, service="API")
db_log   = partial(log, service="Database")

api_log("INFO",  "Request received")
api_log("ERROR", "Timeout after 30s")
db_log ("INFO",  "Connection established")
db_log ("WARN",  "Slow query detected")

# Sorting with a pre-fixed key
data = [{"name": "Charlie", "score": 88},
        {"name": "Alice",   "score": 95},
        {"name": "Bob",     "score": 72}]

get_field = partial(lambda field, d: d[field], "score")
sorted_data = sorted(data, key=get_field, reverse=True)
print("\nSorted by score:")
for emp in sorted_data:
    print(f"  {emp['name']:10} {emp['score']}")

TypeError: log() got multiple values for argument 'service'

## 3. `lru_cache()` — Cache Results Automatically

`@lru_cache` stores (memoizes) the result of expensive function calls.  
If the function is called again with the **same arguments**, it returns the cached result instantly — no recomputation.

**LRU** = Least Recently Used. Once the cache is full, the oldest entry is evicted.

In [ ]:
from functools import lru_cache
import time

# Without caching — recomputes every time
def fib_slow(n):
    if n <= 1:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

# With lru_cache — computes each value only once
@lru_cache(maxsize=128)
def fib_fast(n):
    if n <= 1:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

# Time comparison
start = time.time()
result = fib_slow(35)
print(f"fib_slow(35) = {result}  [{time.time()-start:.3f}s]")

start = time.time()
result = fib_fast(35)
print(f"fib_fast(35) = {result}  [{time.time()-start:.6f}s]")

# Cache info — hits, misses, current size
print("\nCache info:", fib_fast.cache_info())

# Second call — instant (cache hit)
start = time.time()
fib_fast(35)
print(f"Second call  [{time.time()-start:.6f}s] — from cache")

In [ ]:
from functools import lru_cache

# Real-world: caching an expensive database or API lookup
call_count = 0

@lru_cache(maxsize=256)
def get_employee(emp_id):
    global call_count
    call_count += 1
    # Simulated slow DB query
    import time; time.sleep(0.01)
    return {"id": emp_id, "name": f"Employee_{emp_id}", "dept": "Engineering"}

# First calls — hit the "database"
for emp_id in [101, 102, 101, 103, 101, 102]:
    emp = get_employee(emp_id)

print(f"Unique DB calls made: {call_count}")   # 3, not 6
print("Cache info:", get_employee.cache_info())

# Clear cache when data might have changed
get_employee.cache_clear()
print("After clear:", get_employee.cache_info())

## 4. `wraps()` — Preserve Function Metadata in Decorators

When you write a decorator, the wrapper function hides the original function's name and docstring.  
`@wraps(func)` copies the original metadata back.

In [ ]:
from functools import wraps

# Without @wraps — original function identity is lost
def my_decorator_bad(func):
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper

# With @wraps — original identity is preserved
def my_decorator_good(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper

@my_decorator_bad
def greet_bad(name):
    """Greets a person by name."""
    return f"Hello, {name}!"

@my_decorator_good
def greet_good(name):
    """Greets a person by name."""
    return f"Hello, {name}!"

print("Without @wraps:")
print("  __name__:", greet_bad.__name__)      # 'wrapper' — wrong!
print("  __doc__ :", greet_bad.__doc__)       # None — lost!

print("\nWith @wraps:")
print("  __name__:", greet_good.__name__)     # 'greet_good' — correct
print("  __doc__ :", greet_good.__doc__)      # 'Greets a person by name.' — preserved

## Quick Reference

```python
from functools import reduce, partial, lru_cache, wraps

# reduce — fold sequence to one value
reduce(lambda acc, x: acc + x, [1, 2, 3, 4])   # 10
reduce(lambda acc, x: acc * x, [1, 2, 3, 4], 1) # 24 (with initial value)

# partial — pre-fill arguments
double = partial(lambda x, y: x * y, 2)
double(5)   # 10

# lru_cache — memoize expensive calls
@lru_cache(maxsize=128)
def expensive(n): ...
expensive.cache_info()   # CacheInfo(hits=..., misses=..., maxsize=128, currsize=...)
expensive.cache_clear()  # wipe cache

# wraps — use inside every decorator you write
def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

## Interview Questions

1. What does `reduce()` do that `sum()` or `max()` cannot?
2. What is the difference between `partial()` and a lambda that captures variables?
3. What does LRU stand for in `lru_cache`? What happens when the cache is full?
4. Why must arguments to an `lru_cache`-decorated function be **hashable**?
5. What does `@wraps(func)` do and why is it important?
6. How would you use `partial()` to create a multiply-by-10 function from a generic `multiply(a, b)` function?
7. Can `lru_cache` be used on a method that takes a list as argument? Why or why not?

## Practice Exercises

**Easy:**
1. Use `reduce()` to compute the product of all numbers in `[2, 3, 4, 5]`.
2. Create a `double` function using `partial()` applied to `lambda x, factor: x * factor`.

**Medium:**
3. Use `reduce()` to find the longest word in a list of strings.
4. Create an `add_tax` function using `partial()` where tax is fixed at 18%.
5. Cache a function that computes factorials using `lru_cache`. Verify it hits the cache on repeated calls.

**Advanced:**
6. Write a decorator `@timer` that measures function runtime. Use `@wraps` to preserve metadata.
7. Use `lru_cache` to speed up a function that checks if a number is prime. Benchmark with and without it.